## Load the Dataset

In [1]:
from pyspark.sql import SparkSession
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook"

# Initialize Spark Session
spark = SparkSession.builder.appName("LightcastData").getOrCreate()

# Load Data
df = spark.read.option("header", "true").option("inferSchema", "true").option("multiLine","true").option("escape", "\"").csv("data/lightcast_job_postings.csv")
df.createOrReplaceTempView("job_postings")
# Show Schema and Sample Data
# print("---This is Diagnostic check, No need to print it in the final doc---")

# df.printSchema() # comment this line when rendering the submission
# df.show(5)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/18 23:33:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/18 23:34:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Feature Engineering

In [2]:
target_df=spark.sql("""
SELECT
	salary,
	min_years_experience,
	min_edulevels,
	lot_v6_specialized_occupation_name AS occupation
FROM job_postings
GROUP BY 
	salary,
 	min_years_experience,
	min_edulevels,
	occupation
""")
target_df.show()
target_df.createOrReplaceTempView("target_df")

+------+--------------------+-------------+--------------------+
|salary|min_years_experience|min_edulevels|          occupation|
+------+--------------------+-------------+--------------------+
| 98650|                   5|            2|Business Analyst ...|
| 55570|                   1|            2| SAP Analyst / Admin|
| 87037|                   1|            1|        Data Analyst|
|187200|                   7|            2|Enterprise Architect|
|160000|                   5|            2|General ERP Analy...|
|133765|                   5|            2|Oracle Consultant...|
|109200|                   8|            2|        Data Analyst|
|114165|                   6|            2|Oracle Consultant...|
|132500|                   5|            2|Enterprise Architect|
|153920|                   7|           99|        Data Analyst|
|122720|                   3|            0| SAP Analyst / Admin|
|101920|                   4|            2|Oracle Consultant...|
|106000|                 

In [ ]:
## Removing NULLs to clean data
target = 'salary'
key_features = ['min_years_experience', 'min_edulevels', 'occupation']
df_clean = target_df.dropna(subset=[target]+key_features)
df_clean.toPandas().head(5).style.hide(axis="index")

In [15]:
## Impute numeric columns
from pyspark.ml.feature import Imputer
numeric_cols = ["min_years_experience", "min_edulevels", "salary"]
imputer = Imputer(
    inputCols=numeric_cols,
    outputCols=[c + "_imputed" for c in numeric_cols]
).setStrategy("mean")
df_imputed = imputer.fit(df_clean).transform(df_clean)
df_imputed.toPandas().head(5).style.hide(axis="index")

ConnectionRefusedError: [Errno 111] Connection refused

In [ ]:
## Transforming strings to numbers
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import col, count
from pyspark.sql.types import DecimalType, StructField, StringType

occupation_indexer = StringIndexer(inputCol='occupation', outputCol='occupationIndex')
indexed = occupation_indexer.fit(df_clean).transform(df_clean)
column_list = ['occupationIndex']
for column_name in column_list:
    indexed = indexed.withColumn(column_name, col(column_name).cast(DecimalType(18, 2)))
indexed.select("occupation", "occupationIndex"
               ).toPandas().head(5).style.hide(axis="index")


occupation,occupationIndex
Business Analyst (General),5.00
SAP Analyst / Admin,2.00
Data Analyst,0.00
Enterprise Architect,3.00
General ERP Analyst / Consultant,1.00


In [12]:
## One-Hot Encoding categorical variables

from pyspark.ml.feature import OneHotEncoder

encoder = OneHotEncoder(
    inputCols=["occupationIndex"],
    outputCols=["occupationVec"]
)

ohe_model = encoder.fit(indexed)
encoded = ohe_model.transform(indexed)

encoded.select("occupation", "occupationIndex").toPandas().head(5).style.hide(axis="index")

occupation,occupationIndex
Business Analyst (General),5.00
SAP Analyst / Admin,2.00
Data Analyst,0.00
Enterprise Architect,3.00
General ERP Analyst / Consultant,1.00


## Train/Test Split

## Linear Regression

## Generalized Linear Regression Summary

## Diagnostic Plot

## Evaluation